# 04 Export Structures

This notebook writes generated molecules to files that other tools can use.

- **SDF** is useful for cheminformatics, AmberTools input, and later CGenFF/CHARMM-GUI submission because RDKit preserves bond orders and stereochemistry.
- **PDB** is useful for quick inspection in molecular viewers.

During export, hydrogens are added and RDKit creates an initial 3D conformation. This is a starting geometry, not a final equilibrated simulation structure.

## Step 1: Choose output location

Generated structure files go under `examples/output/polymer_structures/`. This keeps PDB/SDF inputs separate from MD outputs such as `benchmark/` and `md_tests/`.

In [1]:
from pathlib import Path

from IPython.display import Markdown

from iphasimulator.workflows import (
    DEFAULT_VALIDATION_TARGETS,
    LARGE_VALIDATION_TARGETS,
    build_validation_molecules,
    describe_molecules,
    export_molecules,
)

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
output_dir = repo_root / "examples" / "output" / "polymer_structures"
output_dir

PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures')

## Step 2: Build and export benchmark structures

The benchmark set includes `P3HB_4`, `P3HB_8`, `P3HO_4`, `P3HO_8`, `P3HDD_4`, and `P3HDD_8`. RDKit writes the SDF files so bond orders and stereochemistry are preserved for later CGenFF/CHARMM-GUI submission.

The returned list shows the exact SDF files written.

In [2]:
benchmark_targets = (
    DEFAULT_VALIDATION_TARGETS[0],
    LARGE_VALIDATION_TARGETS[0],
    DEFAULT_VALIDATION_TARGETS[1],
    LARGE_VALIDATION_TARGETS[1],
    DEFAULT_VALIDATION_TARGETS[2],
    LARGE_VALIDATION_TARGETS[2],
)
molecules = build_validation_molecules(benchmark_targets)
written_paths = export_molecules(molecules, output_dir)
sdf_paths = [path for path in written_paths if path.suffix == ".sdf"]
sdf_paths

[PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/P3HB_4.sdf'),
 PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/P3HB_8.sdf'),
 PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/P3HO_4.sdf'),
 PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/P3HO_8.sdf'),
 PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/P3HDD_4.sdf'),
 PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/polymer_structures/P3HDD_8.sdf')]

## Step 3: Benchmark molecule table

The table uses the RDKit molecule objects built above and reports the isomeric SMILES string plus assigned chiral centres.

In [3]:
def format_chiral_centres(centres):
    return ", ".join(f"{atom_idx}:{label}" for atom_idx, label in centres)


table_rows = [
    {
        "polymer name": row["name"],
        "SMILES string": row["smiles"],
        "chiral centres": format_chiral_centres(row["chiral_centres"]),
    }
    for row in describe_molecules(molecules)
]

markdown_table = [
    "| polymer name | SMILES string | chiral centres |",
    "| --- | --- | --- |",
]
for row in table_rows:
    markdown_table.append(
        f"| {row['polymer name']} | `{row['SMILES string']}` | {row['chiral centres']} |"
    )

Markdown("\n".join(markdown_table))

| polymer name | SMILES string | chiral centres |
| --- | --- | --- |
| P3HB_4 | `C[C@H](CC(=O)O)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)O` | 1:R, 7:R, 13:R, 19:R |
| P3HB_8 | `C[C@H](CC(=O)O)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)OC(=O)C[C@@H](C)O` | 1:R, 7:R, 13:R, 19:R, 25:R, 31:R, 37:R, 43:R |
| P3HO_4 | `CCCCC[C@@H](O)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O` | 1:R, 11:R, 21:R, 31:R |
| P3HO_8 | `CCCCC[C@@H](O)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O[C@H](CCCCC)CC(=O)O` | 1:R, 11:R, 21:R, 31:R, 41:R, 51:R, 61:R, 71:R |
| P3HDD_4 | `CCCCCCCCC[C@@H](O)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O` | 1:R, 15:R, 29:R, 43:R |
| P3HDD_8 | `CCCCCCCCC[C@@H](O)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O[C@H](CCCCCCCCC)CC(=O)O` | 1:R, 15:R, 29:R, 43:R, 57:R, 71:R, 85:R, 99:R |

## Command-line equivalent

For repeated runs, the command-line script is faster than opening Jupyter:

```bash
PYTHONPATH=src python examples/generate_validation_structures.py
```

To include the larger n = 8 validation molecules:

```bash
PYTHONPATH=src python examples/generate_validation_structures.py --include-large
```